# 1) Import data and libraries

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../modules')

import matplotlib.pyplot as plt
import matplotlib.animation as anim
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.signal import butter, filtfilt
plt.rcParams['figure.dpi'] = 150
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['animation.embed_limit'] = 2**128


from Class_sem2dpack import *
from Stage_module import *

# Directory name with output files
direct1 = "C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/OUTPUT/Stage_ISS_v10_verticalinci_stations"
is_overburden = False
fmin, fmax = 0.01, 50.0
SEM = sem2dpack(direct1)

# 2) Read Data

### Read SEM2DPACK Fault Data

In [ ]:
read_fault_testing(SEM, ftag=5)
BC_bottom = SEM.fault

read_fault_testing(SEM, ftag=7)
BC_right = SEM.fault

read_fault_testing(SEM, ftag=9)
BC_left = SEM.fault

vector_orientations = {}
vector_orientations['Left'] = -1, -1
vector_orientations['Bottom'] = -1, +1
vector_orientations['Right'] = +1, +1

X_coord_L = BC_left['x']
X_coord_B = BC_bottom['x']
X_coord_R = BC_right['x']

Z_coord_L = BC_left['z']
Z_coord_B = BC_bottom['z']
Z_coord_R = BC_right['z']

Slip_1_L = BC_left['Slip_1']
Slip_1_B = BC_bottom['Slip_1']
Slip_1_R = BC_right['Slip_1']

Slip_2_L = BC_left['Slip_2']
Slip_2_B = BC_bottom['Slip_2']
Slip_2_R = BC_right['Slip_2']

time_fault = BC_left['Time']



fault_components = {}
fault_components["Left"] = X_coord_L, Z_coord_L, Slip_1_L, Slip_2_L, 'z'
fault_components["Bottom"] = X_coord_B, Z_coord_B, Slip_1_B, Slip_2_B, 'x'
fault_components["Right"] = X_coord_R, Z_coord_R, Slip_1_R, Slip_2_R, 'z'


    

### Read SEM2DPACK Station Data

In [ ]:
SEM.read_seismo('x')
Time_stations = SEM.time
Vx = SEM.velocity[:,:]
Ux = np.zeros(Vx.shape)
for i in range(Vx.shape[1]):
    V = Vx[:,i]
    Ux[:,i] = compute_displacement(V, Time_stations)

SEM.read_seismo('z')
Vz = SEM.velocity[:,:]
Uz = np.zeros(Vz.shape)
for i in range(Vz.shape[1]):
    V = Vz[:,i]
    Uz[:,i] = compute_displacement(V, Time_stations)

XSTA, ZSTA = SEM.rcoord[:,0], SEM.rcoord[:,1]

components = {}
components['x'] = Time_stations, Ux, Vx, 'blue'
components['z'] = Time_stations, Uz, Vz, 'red'


### Read FLAC Data

In [ ]:
import ast 

def filter(signal, fc, fe, order=4):
    b, a = butter(order, fc / (0.5 * fe), btype='low')
    return filtfilt(b, a, signal)    

def read_FLAC_TXT_signal(file_path, fmax):
    """
    Reads a signal from a TXT file and returns the data as a numpy array.
    """
    data = pd.read_table(file_path, skiprows=2, sep='\s+')
    time = data.iloc[:, 0].to_numpy()
    signal = data.iloc[:, 1].to_numpy()
    dt = time[1] - time[0]
    signal = filter(signal, fmax, fe=1/dt)
    return time, signal

def read_fault_signal(file_path, fmax):
    data = pd.read_table(file_path)
    time = data.iloc[:, 0].to_numpy()
    normal_disp = data.iloc[:, 1].to_numpy()
    shear_disp = data.iloc[:, 2].to_numpy()
    shear_disp = np.array([ast.literal_eval(s.strip()) for s in shear_disp], dtype=float)
    if np.all(shear_disp[:,0] == 0) :
        Ux = normal_disp
        Uz = shear_disp[:,2]
    else:
        Ux = shear_disp[:,0]
        Uz = normal_disp
    dt = time[1] - time[0]
    print(dt)
    Ux = filter(Ux, fmax, fe=1/dt)
    Uz = filter(Uz, fmax, fe=1/dt)
    return time, Ux, Uz

def read_fault_signal_test(file_path, fmax):
    data = pd.read_table(file_path)
    time = data.iloc[:, 0].to_numpy()
    normal_disp = data.iloc[:, 1].to_numpy()
    shear_disp = data.iloc[:, 2].to_numpy()
    dt = time[1] - time[0]
    normal = filter(normal_disp, fmax, fe=1/dt)
    shear = filter(shear_disp, fmax, fe=1/dt)
    return time, normal, shear

# time_fault_FLAC, Ux_L, Uz_L = read_fault_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Left_Interface.txt", 4)
# _, Ux_B, Uz_B = read_fault_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Bottom_Interface.txt", 4)
# _, Ux_R, Uz_R = read_fault_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Right_Interface.txt", 4)


time_fault_FLAC, Ux_L, Uz_L = read_fault_signal_test("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Left_Interface.txt", 4)
_, Uz_B, Ux_B = read_fault_signal_test("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Bottom_Interface.txt", 4)
_, Ux_R, Uz_R = read_fault_signal_test("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Right_Interface.txt", 4)

time_stations_FLAC, Ux_L_S = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Ux_Left.txt", 4)
_, Uz_L_S = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Uz_Left.txt", 4)
_, Ux_B_S = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Ux_Bottom.txt", 4)
_, Uz_B_S = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Uz_Bottom.txt", 4)
_, Ux_R_S = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Ux_Right.txt", 4)
_, Uz_R_S = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Uz_Right.txt", 4)

time_input, Vx_input = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Input.txt", 4)



# 3) Preview

### Plot Input Signal

In [ ]:
#Load input signal
input_signal_path = SEM.directory + '/' + 'SourcesTime_sem2d.tab'
input_signal = np.genfromtxt(open(input_signal_path,'r'))
input_time = input_signal[:,0]
input_velocity = input_signal[:,1]

file = open("Input_Signal_Table.txt", 'w+')
file.write("Input signal\n")
file.write(str(input_time.size) + "  " + str(input_time[1]-input_time[0]) + "\n")
for i in input_velocity:
    file.write(str(i) + "\n")
file.close()

#Create figure
fig = plt.figure()
fig.subplots_adjust(wspace=0.5)
fig.suptitle("Input signal")

#Create velocity plot
ax_V = fig.add_subplot(1,2,1)
ax_V.set_xlabel("Time (s)")
ax_V.set_ylabel("Velocity (m/s)")
ax_V.set_title("Input velocity")
ax_V.grid(True)
ax_V.plot(input_time, input_velocity, 'b-', label='SEM2DPACK input')
ax_V.plot(time_input, Vx_input, 'r--', label='FLAC input')
ax_V.legend()

#Create displacement plot
input_displacement = compute_displacement(input_velocity,input_time)
input_displacement_FLAC = compute_displacement(Vx_input,time_input)

ax_D = fig.add_subplot(1,2,2)
ax_D.set_xlabel("Time (s)")
ax_D.set_ylabel("Displacement (m)")
ax_D.set_title("Input displacement")
ax_D.grid(True)
ax_D.plot(input_time, input_displacement, 'b-', label='SEM2DPACK input')
ax_D.plot(time_input, input_displacement_FLAC, 'r--', label='FLAC input')
ax_D.legend()




### Drawing of the example with the stations

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(4, 4))

#Desired stations
stations_left = [get_nearest_station(-12,0, XSTA, ZSTA),get_nearest_station(-12,-15, XSTA, ZSTA),get_nearest_station(-12,-30, XSTA, ZSTA)]
stations_bottom = [get_nearest_station(-10,-32, XSTA, ZSTA),get_nearest_station(0,-32, XSTA, ZSTA),get_nearest_station(10,-32, XSTA, ZSTA)]
stations_right = [get_nearest_station(12,0, XSTA, ZSTA),get_nearest_station(12,-15, XSTA, ZSTA),get_nearest_station(12,-30, XSTA, ZSTA)]
stations = np.concatenate([stations_left, stations_bottom, stations_right])

stations_dic = {}
stations_dic["Left"] = stations_left
stations_dic["Bottom"] = stations_bottom
stations_dic["Right"] = stations_right

#Desired fault locations
stations_left_fault = [12, 7, 0]
stations_bottom_fault = [0, 4, 8]
stations_right_fault = [12, 7, 0]

stations_fault_dic = {}
stations_fault_dic["Left"] = stations_left_fault
stations_fault_dic["Bottom"] = stations_bottom_fault
stations_fault_dic["Right"] = stations_right_fault

#Draw the plot
draw_example(ax)
ax.scatter(XSTA, ZSTA, marker='v')
ax.scatter(XSTA[stations], ZSTA[stations], marker='v')
ax.set_title("Drawing of the stations")
ax.set_ylim(-80,35)


# 4) Post-Processing

### Comparison between station data

In [ ]:

def plot_comparison(comp, interf):
    plt.xlabel("Time (s)")
    plt.ylabel("Displacement (m)")
    plt.grid(True)
    if comp == 'x':
        if interf == 'Left':
            index = get_nearest_station(-12,0, XSTA, ZSTA)
            plt.plot(Time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_2_L[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Ux_L_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Ux_L, label="FLAC interface", c='orange')
        elif interf == 'Bottom':
            index = get_nearest_station(-0,-32, XSTA, ZSTA)
            plt.plot(Time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_1_B[1,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Ux_B_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Ux_B, label="FLAC interface", c='orange')
        elif interf == 'Right':
            index = get_nearest_station(12,0, XSTA, ZSTA)
            plt.plot(Time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_2_R[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Ux_R_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Ux_R, label="FLAC interface", c='orange')
    elif comp == 'z':
        if interf == 'Left':
            index = get_nearest_station(-12,0, XSTA, ZSTA)
            plt.plot(Time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, -Slip_1_L[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Uz_L_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Uz_L, label="FLAC interface", c='orange')
        elif interf == 'Bottom':
            index = get_nearest_station(0,-32, XSTA, ZSTA)
            plt.plot(Time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_2_B[1,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Uz_B_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Uz_B, label="FLAC interface", c='orange')
        elif interf == 'Right':
            index = get_nearest_station(12,0, XSTA, ZSTA)
            plt.plot(Time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_1_R[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Uz_R_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Uz_R, label="FLAC interface", c='orange')
    plt.legend()
    plt.title(f"{comp} displacement at {interf} Station (30°)")
    plt.show()

plot_comparison('x', 'Left')
plot_comparison('x', 'Bottom')
plot_comparison('x', 'Right')
plot_comparison('z', 'Left')
plot_comparison('z', 'Bottom')
plot_comparison('z', 'Right')

     





## Multiple stations

In [ ]:
time_S_FLAC, Ux_40_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_40_X.txt", 4)
_, Uz_40_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_40_Z.txt", 4)
_, Ux_50_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_50_X.txt", 4)
_, Uz_50_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_50_Z.txt", 4)
_, Ux_60_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_60_X.txt", 4)
_, Uz_60_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_60_Z.txt", 4)
_, Ux_70_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_70_X.txt", 4)
_, Uz_70_FLAC = read_FLAC_TXT_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_70_Z.txt", 4)

Ux_40_SEM = Ux[:,get_nearest_station(0,-40, XSTA, ZSTA)]
Uz_40_SEM = Uz[:,get_nearest_station(0,-40, XSTA, ZSTA)]
Ux_50_SEM = Ux[:,get_nearest_station(0,-50, XSTA, ZSTA)]
Uz_50_SEM = Uz[:,get_nearest_station(0,-50, XSTA, ZSTA)]
Ux_60_SEM = Ux[:,get_nearest_station(0,-60, XSTA, ZSTA)]
Uz_60_SEM = Uz[:,get_nearest_station(0,-60, XSTA, ZSTA)]
Ux_70_SEM = Ux[:,get_nearest_station(0,-70, XSTA, ZSTA)]
Uz_70_SEM = Uz[:,get_nearest_station(0,-70, XSTA, ZSTA)]


def plot_stations(comp):
    fig = plt.figure()
    ax_FLAC = fig.add_subplot(2, 1, 1)
    ax_SEM = fig.add_subplot(2, 1, 2)
    fig.suptitle(f"{comp.capitalize()} Displacements of stations at X=0m")
    if comp == 'x':
        ax_FLAC.plot(time_S_FLAC, Ux_40_FLAC, label="FLAC Station at Z=-40m")
        ax_FLAC.plot(time_S_FLAC, Ux_50_FLAC, label="FLAC Station at Z=-50m")
        ax_FLAC.plot(time_S_FLAC, Ux_60_FLAC, label="FLAC Station at Z=-60m")
        ax_FLAC.plot(time_S_FLAC, Ux_70_FLAC, label="FLAC Station at Z=-70m")
        # ax_FLAC.set_xlim(2.5,10)
        # ax_FLAC.set_ylim(0.224,0.226)
        # ax_FLAC.set_xlim(1.75,2.25)
        # ax_FLAC.set_ylim(0.08,0.12)
        ax_SEM.plot(Time_stations, Ux_40_SEM, label="SEM2DPACK Station at Z=-40m")
        ax_SEM.plot(Time_stations, Ux_50_SEM, label="SEM2DPACK Station at Z=-50m")
        ax_SEM.plot(Time_stations, Ux_60_SEM, label="SEM2DPACK Station at Z=-60m")
        ax_SEM.plot(Time_stations, Ux_70_SEM, label="SEM2DPACK Station at Z=-70m")
    elif comp == 'z':
        ax_FLAC.plot(time_S_FLAC, Uz_40_FLAC, label="FLAC Station at Z=-40m")
        ax_FLAC.plot(time_S_FLAC, Uz_50_FLAC, label="FLAC Station at Z=-50m")
        ax_FLAC.plot(time_S_FLAC, Uz_60_FLAC, label="FLAC Station at Z=-60m")
        ax_FLAC.plot(time_S_FLAC, Uz_70_FLAC, label="FLAC Station at Z=-70m")
        ax_SEM.plot(Time_stations, Uz_40_SEM, label="SEM2DPACK Station at Z=-40m")
        ax_SEM.plot(Time_stations, Uz_50_SEM, label="SEM2DPACK Station at Z=-50m")
        ax_SEM.plot(Time_stations, Uz_60_SEM, label="SEM2DPACK Station at Z=-60m")
        ax_SEM.plot(Time_stations, Uz_70_SEM, label="SEM2DPACK Station at Z=-70m")
    fig.supxlabel("Time (s)")
    fig.supylabel("Displacement (m)")
    ax_FLAC.grid(True)
    ax_SEM.grid(True)
    ax_FLAC.legend()
    ax_SEM.legend()
    plt.show()

plot_stations('x')
plot_stations('z')

## Comparison between fault data

In [ ]:

def plot_comparison(comp, interf):
    plt.xlabel("Time (s)")
    plt.ylabel("Displacement (m)")
    plt.grid(True)
    if comp == 'x':
        if interf == 'Left':
            index = get_nearest_station(-12,0, XSTA, ZSTA)
            # plt.plot(Time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            plt.plot(time_fault, Slip_2_L[7,:], label="SEM2DPACK Interface", c='green')
            # plt.plot(time_stations_FLAC, Ux_L_S, label="FLAC Station", c='purple')  
            plt.plot(time_fault_FLAC, Ux_L, label="FLAC interface", c='orange')
        elif interf == 'Bottom':
            index = get_nearest_station(-0,-32, XSTA, ZSTA)
            # plt.plot(Time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            plt.plot(time_fault, Slip_1_B[1,:], label="SEM2DPACK Interface", c='green')
            # plt.plot(time_stations_FLAC, Ux_B_S, label="FLAC Station", c='purple')  
            plt.plot(time_fault_FLAC, Ux_B, label="FLAC interface", c='orange')
        elif interf == 'Right':
            index = get_nearest_station(12,0, XSTA, ZSTA)
            # plt.plot(Time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            plt.plot(time_fault, Slip_2_R[7,:], label="SEM2DPACK Interface", c='green')
            # plt.plot(time_stations_FLAC, Ux_R_S, label="FLAC Station", c='purple')  
            plt.plot(time_fault_FLAC, Ux_R, label="FLAC interface", c='orange')
    elif comp == 'z':
        if interf == 'Left':
            index = get_nearest_station(-12,0, XSTA, ZSTA)
            # plt.plot(Time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            plt.plot(time_fault, -Slip_1_L[7,:], label="- SEM2DPACK Interface", c='green')
            # plt.plot(time_stations_FLAC, Uz_L_S, label="FLAC Station", c='purple')  
            plt.plot(time_fault_FLAC, Uz_L, label="FLAC interface", c='orange')
        elif interf == 'Bottom':
            index = get_nearest_station(0,-32, XSTA, ZSTA)
            # plt.plot(Time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            plt.plot(time_fault, Slip_2_B[1,:], label="SEM2DPACK Interface", c='green')
            # plt.plot(time_stations_FLAC, Uz_B_S, label="FLAC Station", c='purple')  
            plt.plot(time_fault_FLAC, Uz_B, label="FLAC interface", c='orange')
        elif interf == 'Right':
            index = get_nearest_station(12,0, XSTA, ZSTA)
            # plt.plot(Time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            plt.plot(time_fault, Slip_1_R[7,:], label="SEM2DPACK Interface", c='green')
            # plt.plot(time_stations_FLAC, Uz_R_S, label="FLAC Station", c='purple')  
            plt.plot(time_fault_FLAC, Uz_R, label="FLAC interface", c='orange')
    plt.legend()
    plt.title(f"{comp.capitalize()} displacement at {interf.capitalize()} Interface (kn = 2.5e7 ; ks = 1e9 ; phi = 16.7°)")
    plt.show()

plot_comparison('x', 'Left')
plot_comparison('x', 'Bottom')
plot_comparison('x', 'Right')
plot_comparison('z', 'Left')
plot_comparison('z', 'Bottom')
plot_comparison('z', 'Right')

     





In [ ]:

np.rad2deg(np.arctan(0.3))

In [ ]:
cp = 700
cs = 300
rho = 2000

ratio = cp/cs

nu = (ratio**2 - 2)/(2*(ratio**2 - 1))
E = rho*(3*cp**2 - 4*cs**2)/(ratio**2 - 1)

print(f"nu = {nu:.3e}")
print(f"E = {E:.3e} Pa")